# 02 — 单变量探索性分析

> 🎯 **适用场景**: 逐字段理解数据分布，发现异常和模式
> ⏱️ **预计学习时长**: 60 分钟
> 📌 **核心问题**: 每个字段"长什么样"？数值分布正态还是偏态？分类变量有几个类？时间趋势如何？

---

## 本 Notebook 结构

1. **数值型** → 直方图 / 密度曲线 / 箱线图 → 偏度检测
2. **分类型** → 频次统计 / 条形图 / 饼图
3. **时间型** → 月度趋势 / 周度模式 / 节假日分析


In [ ]:
# ═══════════════════════════════════════════
# 环境准备（沿用 01_notebook 数据）
# ═══════════════════════════════════════════

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei']
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.dpi'] = 150

pd.set_option('display.max_columns', 100)

# ── 加载数据 ──
DATA_DIR = "../data"
df_orders   = pd.read_csv(f"{DATA_DIR}/olist_orders_dataset.csv", parse_dates=[
    'order_purchase_timestamp', 'order_approved_at',
    'order_delivered_carrier_date', 'order_delivered_customer_date',
    'order_estimated_delivery_date'])
df_items    = pd.read_csv(f"{DATA_DIR}/olist_order_items_dataset.csv")
df_payments = pd.read_csv(f"{DATA_DIR}/olist_order_payments_dataset.csv")
df_products = pd.read_csv(f"{DATA_DIR}/olist_products_dataset.csv")
df_reviews  = pd.read_csv(f"{DATA_DIR}/olist_order_reviews_dataset.csv")

# ── 合并常用关联列 ──
df_items['shipping_limit_date'] = pd.to_datetime(df_items['shipping_limit_date'])

# orders + items 一键 Join（后续分析高频使用）
df_merged = df_orders.merge(df_items, on='order_id', how='inner')
# orders + payments
df_ord_pay = df_orders.merge(df_payments, on='order_id', how='inner')

print("✅ 数据加载完成")
print(f"  merged (orders+items): {df_merged.shape}")
print(f"  ord_pay (orders+payments): {df_ord_pay.shape}")


## 一、数值型特征分析

### 1.1 价格分布

📌 价格是电商最核心的数值字段。我们关注分布形态和极端值。

In [ ]:
# 📌 价格分布 — 直方图 + 密度曲线 + 箱线图
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# 原始分布
sns.histplot(df_items['price'], bins=100, kde=True, color='#1f77b4', ax=axes[0, 0])
axes[0, 0].set_title('Price 分布 (原始)', fontsize=13, fontweight='bold')
axes[0, 0].set_xlabel('price (R$)')

# 对数变换后分布 (📌 长尾数据常用技巧)
sns.histplot(np.log1p(df_items['price']), bins=50, kde=True, color='#ff7f0e', ax=axes[0, 1])
axes[0, 1].set_title('log(Price+1) 分布', fontsize=13, fontweight='bold')

# 箱线图
sns.boxplot(x=df_items['price'], color='#2ca02c', ax=axes[1, 0])
axes[1, 0].set_title('Price 箱线图', fontsize=13, fontweight='bold')

# IQR 裁剪后的分布 (>0, <P99)
q99 = df_items['price'].quantile(0.99)
sns.histplot(df_items[(df_items['price']>0)&(df_items['price']<q99)]['price'],
             bins=50, color='#d62728', ax=axes[1, 1])
axes[1, 1].set_title(f'Price 分布 (0 ~ P99={q99:.0f})', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.savefig('../outputs/02_price_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

# 📌 关键统计量
stats = df_items['price'].describe(percentiles=[.25, .5, .75, .9, .95, .99])
print("📌 price 关键统计:")
print(f"  均值: R$ {stats['mean']:.2f}")
print(f"  中位数: R$ {stats['50%']:.2f}")
print(f"  P90: R$ {stats['90%']:.0f}  |  P95: R$ {stats['95%']:.0f}")
print(f"  P99: R$ {stats['99%']:.0f}  |  max: R$ {stats['max']:.0f}")
print(f"  ⚡ 偏度: {df_items['price'].skew():.2f} (|skew|>1 → 长尾分布)")

# 💭 反思：为什么价格呈严重右偏分布？这对建模有什么影响？


In [ ]:
# 📌 运费分布
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.histplot(df_items['freight_value'], bins=50, kde=True, color='#9467bd', ax=axes[0])
axes[0].set_title('运费分布', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Freight Value (R$)')

sns.boxplot(x=df_items['freight_value'], color='#8c564b', ax=axes[1])
axes[1].set_title('运费箱线图', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.show()

# 📌 运费 vs 商品价格的比例
df_items['freight_ratio'] = df_items['freight_value'] / df_items['price']
print(f"📌 运费占商品价格比例:  均值={df_items['freight_ratio'].mean():.1%}  "
      f"中位数={df_items['freight_ratio'].median():.1%}")
# 💭 反思：有些商品运费比商品本身还贵，这是什么场景？


### 1.2 支付金额分布

In [ ]:
# 📌 支付金额分布（每笔支付记录）
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.histplot(df_payments['payment_value'], bins=50, kde=True, color='#1f77b4', ax=axes[0])
axes[0].set_title('支付金额分布（每笔支付）', fontsize=13, fontweight='bold')

sns.boxplot(x=df_payments['payment_value'], color='#ff7f0e', ax=axes[1])
axes[1].set_title('支付金额箱线图', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.show()

print(f"📌 payment_value 统计:  mean=R$ {df_payments['payment_value'].mean():.2f}  "
      f"median=R$ {df_payments['payment_value'].median():.2f}")


## 二、分类型特征分析

### 2.1 订单状态分布

📌 订单状态构成一个天然的漏斗：created → delivered。非"delivered"的订单是问题订单。

In [ ]:
# 📌 订单状态分布
status_counts = df_orders['order_status'].value_counts()

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# 环形图
colors = ['#4CAF50','#FF9800','#F44336','#2196F3','#9C27B0','#FF5722','#795548','#607D8B']
wedges, texts, autotexts = axes[0].pie(
    status_counts.values, labels=None,
    autopct='%1.1f%%', startangle=90,
    colors=colors, pctdistance=0.85,
    textprops={'fontsize': 10})

axes[0].set_title('订单状态分布（环形图）', fontsize=13, fontweight='bold')
# 中心空白 → 环形
centre_circle = plt.Circle((0, 0), 0.60, fc='white')
axes[0].add_artist(centre_circle)

# 条形图
status_counts.plot(kind='barh', color=colors, ax=axes[1])
axes[1].set_title('订单状态分布（条形图）', fontsize=13, fontweight='bold')
axes[1].set_xlabel('订单数')

plt.legend(wedges, status_counts.index, title='订单状态', loc='center left', bbox_to_anchor=(1, 0.5))
plt.tight_layout()
plt.savefig('../outputs/02_order_status.png', dpi=150, bbox_inches='tight')
plt.show()

print("📌 订单状态统计:")
print(status_counts.to_string())

# 📌 计算有效订单率 = delivered/(delivered+shipped+processing+approved)
valid_rate = status_counts.get('delivered',0) / status_counts.sum() * 100
print(f"\n📌 有效（delivered）率: {valid_rate:.1f}%")


### 2.2 支付方式偏好

📌 巴西的支付方式和中国差异很大，了解这个对业务决策有启发。

In [ ]:
# 📌 支付方式分布
pay_type = df_payments['payment_type'].value_counts()

fig, ax = plt.subplots(figsize=(10, 6))
bars = pay_type.plot(kind='bar', color=['#1f77b4','#ff7f0e','#2ca02c','#d62728','#9467bd'], ax=ax)
ax.set_title('支付方式分布', fontsize=14, fontweight='bold')
ax.set_ylabel('订单数')
ax.set_xlabel('')
plt.xticks(rotation=20)

for i, (v, pct) in enumerate(zip(pay_type.values, pay_type.values/pay_type.sum()*100)):
    ax.text(i, v + max(pay_type.values)*0.01, f'{pct:.1f}%', ha='center', fontsize=11)

plt.tight_layout()
plt.show()

print(pay_type)
print("\n💭 反思: boleto（银行票据）占比这么高，跟巴西的信用卡渗透率有关吗？")


### 2.3 产品类别 Top 15

📌 Olist 的产品类别名是葡萄牙语，我们对照翻译表转成英文/中文。

In [ ]:
# 📌 加载翻译表，将葡萄牙语类别转英文
df_trans = pd.read_csv(f"{DATA_DIR}/product_category_name_translation.csv")

# products 无缺失类别用翻译表映射
df_products = df_products.merge(df_trans, on='product_category_name', how='left')
df_products['category_en'] = df_products['product_category_name_english'].fillna(
    df_products['product_category_name'])

# 合并 items → 得到每个订单项的类别
df_items_full = df_items.merge(
    df_products[['product_id', 'category_en']],
    on='product_id', how='left')

# 📌 Top 15 品类销量
top_cat = df_items_full['category_en'].value_counts().head(15)

fig, ax = plt.subplots(figsize=(10, 8))
top_cat.sort_values().plot(kind='barh', color=sns.color_palette("viridis", 15), ax=ax)
ax.set_title('Top 15 产品品类（销量）', fontsize=14, fontweight='bold')
ax.set_xlabel('销量')

for i, v in enumerate(top_cat.sort_values().values):
    ax.text(v + 10, i, f'{v:,}', va='center', fontsize=9)

plt.tight_layout()
plt.savefig('../outputs/02_top_categories.png', dpi=150, bbox_inches='tight')
plt.show()

# 📌 集中度分析
print(f"Top 5 品类占总销量: {top_cat.head(5).sum()/len(df_items_full)*100:.1f}%")
print(f"Top 15 品类占总销量: {top_cat.sum()/len(df_items_full)*100:.1f}%")


## 三、时间特征分析

### 3.1 月度订单量趋势

📌 从月度趋势看增长态势和季节性。

In [ ]:
# 📌 月度订单量趋势
df_orders['year_month'] = df_orders['order_purchase_timestamp'].dt.to_period('M')
monthly_orders = df_orders.groupby('year_month').size()
monthly_orders.index = monthly_orders.index.astype(str)

fig, ax = plt.subplots(figsize=(16, 6))
monthly_orders.plot(kind='line', marker='o', linewidth=2, markersize=6, color='#1f77b4', ax=ax)

# 标注最高和最低点
max_idx = monthly_orders.idxmax()
min_idx = monthly_orders.idxmin()
ax.annotate(f'峰值: {monthly_orders[max_idx]:,} 单',
            xy=(monthly_orders.index.get_loc(max_idx), monthly_orders[max_idx]),
            xytext=(10, 20), textcoords='offset points', fontsize=11,
            arrowprops=dict(arrowstyle='->', color='#d62728'))

ax.set_title('月度订单量趋势 (2016.09 ~ 2018.10)', fontsize=15, fontweight='bold')
ax.set_ylabel('订单量')
ax.grid(True, alpha=0.3)
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('../outputs/02_monthly_orders.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"📌 月度订单量: max={monthly_orders[max_idx]:,} | min={monthly_orders[min_idx]:,}")
print(f"   2017 总订单: {monthly_orders['2017-01':'2017-12'].sum():,}")
print(f"   2018 前9月:   {monthly_orders['2018-01':'2018-09'].sum():,}")

# 💭 反思：2018年8月后的骤降是季节因素还是数据不完整？


In [ ]:
# 📌 星期分布：巴西人星期几最爱网购？
df_orders['day_of_week'] = df_orders['order_purchase_timestamp'].dt.dayofweek  # 0=Mon
df_orders['day_name'] = df_orders['order_purchase_timestamp'].dt.day_name()

dow_order = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
dow_counts = df_orders['day_name'].value_counts().reindex(dow_order)

fig, ax = plt.subplots(figsize=(10, 6))
colors_dow = ['#d62728' if d in ['Saturday','Sunday'] else '#1f77b4' for d in dow_order]
dow_counts.plot(kind='bar', color=colors_dow, ax=ax)
ax.set_title('星期分布：哪天最爱下单？', fontsize=14, fontweight='bold')
ax.set_xlabel('')
ax.set_ylabel('订单数')
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

print("📌 每天订单占比:")
for d, c in dow_counts.items():
    print(f"  {d:12s}: {c:>6,} ({c/dow_counts.sum()*100:.1f}%)")


## 📝 康奈尔笔记联动区

### 左侧栏（核心概念）
- **对数变换**: 处理长尾分布的常用方法
- **订单状态漏斗**: 只有"delivered"的订单才算有效成交
- **周度模式**: 工作日 vs 周末的订单差异

### 右侧栏（反思问题）
> 💭 为什么 boleto 在巴西这么流行？对商家回款周期有什么影响？
> 💭 价格长尾分布中，最高价商品是否合理？需要抽样验证
> 💭 月度趋势中 2018 Q4 的下降是真的下滑还是数据不完整？

### 底部栏（行动清单）
- [ ] 验证 price > P99.9 的商品是否合理
- [ ] 对比巴西节假日和订单峰值的关系
- [ ] 到 project01_reflections.md 记录 2 个洞察
